# Module 06 — Mixture of Experts (notebook)

Walkthrough of [`experts.py`](experts.py). We'll:

1. Build an `MoEFFN`, run forward, inspect the router's behavior at init.
2. Demonstrate the **load-imbalance failure mode** — train an MoE without balancing on a synthetic 8-mode task and watch one expert eat all the traffic.
3. Repeat with **aux-loss-free balancing** (DeepSeek-V3's mechanism) and the **classical aux-loss** (Switch / GShard). Compare loss curves and utilization.
4. Quantify the MoE win: total vs active parameter count.
5. Run an `MoETransformerBlock` end-to-end.

**Compute:** CPU is enough.  
**Time:** ~10 minutes.

In [ ]:
import sys, pathlib
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from experts import Expert, Router, MoEFFN, MoETransformerBlock

# Module 04 + 05 components, for the end-to-end block at the end.
sys.path.append(str(pathlib.Path.cwd().parent / "04-attention"))
sys.path.append(str(pathlib.Path.cwd().parent / "05-transformer-block"))
from attention import GroupedQueryAttention
from rope import precompute_freqs_cis

torch.manual_seed(0)
print("torch:", torch.__version__)

## 1. MoEFFN forward and router behavior at init

Build a small MoE — 8 experts, top-2 routing, no shared expert for clarity — and verify the forward shape. At initialization the router weights are tiny random values, so utilization should be roughly uniform: each expert sees about $k/N = 2/8 = 0.25$ of the tokens.

In [ ]:
B, T, d_model = 4, 64, 128
n_experts, top_k = 8, 2

moe = MoEFFN(
    d_model=d_model, n_experts=n_experts, top_k=top_k,
    d_ffn_expert=64, n_shared_experts=0,  # no shared, for a clean utilization plot
)
x = torch.randn(B, T, d_model)
out, info = moe(x)

print(f"input  shape: {tuple(x.shape)}")
print(f"output shape: {tuple(out.shape)}")
print(f"utilization:  {[f'{u:.3f}' for u in info.utilization.tolist()]}")
print(f"target:       {top_k / n_experts:.3f}  (top_k / n_experts)")
print(f"max/min util ratio: {info.utilization.max().item() / info.utilization.min().item():.2f}")

Already at init the utilization is not perfectly uniform — the top expert sees a few times more traffic than the bottom one, even with random router weights and random inputs. That's the seed of the collapse problem. Once SGD takes over, the imbalance compounds.

## 2. Synthetic 8-mode task — demonstrating load imbalance

Design a task where 8 experts *should* specialize, one per mode:

- The input $x$ has a one-hot "mode" feature in the first 8 dimensions (the rest is random noise).
- The target $y$ is a mode-specific linear function of the noise.
- A perfectly-balanced router learns to route each mode to its own expert; an unbalanced one routes everything to one or two experts and gets the other modes wrong.

This is a clean test of whether the balancing mechanism prevents collapse. We'll train three identical MoEs:

1. `balancing="none"` — baseline, expect collapse.
2. `balancing="aux_loss"` — classical Switch / GShard balancing.
3. `balancing="aux_loss_free"` — DeepSeek-V3's bias-update mechanism.

In [ ]:
def synthetic_batch(batch_size, d_model, n_modes=8, mode_noise=2.0, seed=None):
    """8-mode task with a NOISY mode signal — so the router has to actually learn.

    Each input has a mode 0..7. The first 8 dims are a one-hot of the mode
    PLUS heavy Gaussian noise; the rest is random noise. Targets are
    mode-specific linear maps of the noise. The mode_noise factor controls
    how hard routing is: 0 means clean one-hot (trivial), 2.0 means the
    router can barely tell modes apart from raw input — it has to learn
    a non-trivial mapping from input to expert.
    """
    g = torch.Generator().manual_seed(seed) if seed is not None else None
    mode = torch.randint(0, n_modes, (batch_size,), generator=g)
    mode_oh = F.one_hot(mode, n_modes).float()
    mode_signal = mode_oh + mode_noise * torch.randn(batch_size, n_modes, generator=g)
    noise = torch.randn(batch_size, d_model - n_modes, generator=g)
    x = torch.cat([mode_signal, noise], dim=-1)
    target = torch.zeros(batch_size, d_model)
    for m in range(n_modes):
        mask = mode == m
        if mask.any():
            torch.manual_seed(100 + m)
            proj = torch.randn(d_model - n_modes, d_model)
            target[mask] = noise[mask] @ proj
    return x.unsqueeze(1), target.unsqueeze(1)


def train_moe(balancing, n_steps=400, batch_size=128, seed=0):
    torch.manual_seed(seed)
    moe = MoEFFN(
        d_model=64, n_experts=8, top_k=1, d_ffn_expert=32,
        n_shared_experts=0, balancing=balancing,
    )
    opt = torch.optim.Adam(moe.parameters(), lr=3e-3)

    util_history = []
    loss_history = []

    for step in range(n_steps):
        x, y = synthetic_batch(batch_size, 64)
        out, info = moe(x)
        loss = F.mse_loss(out, y)
        if info.aux_loss is not None:
            loss = loss + info.aux_loss
        opt.zero_grad(); loss.backward(); opt.step()

        # Aux-loss-free: update the router bias AFTER optimizer.step
        moe.router.update_bias(info.utilization)

        util_history.append(info.utilization.detach().tolist())
        loss_history.append(loss.item())

    return moe, torch.tensor(util_history), torch.tensor(loss_history)

In [ ]:
print("Training no-balancing baseline...")
_, util_none, loss_none = train_moe("none")
print("Training with classical aux-loss balancing...")
_, util_aux,  loss_aux  = train_moe("aux_loss")
print("Training with aux-loss-free balancing (DeepSeek-V3)...")
_, util_free, loss_free = train_moe("aux_loss_free")

print(f"\nFinal utilization (target = 0.125):")
print(f"  no balancing:     {[f'{u:.3f}' for u in util_none[-1].tolist()]}")
print(f"  aux_loss:         {[f'{u:.3f}' for u in util_aux [-1].tolist()]}")
print(f"  aux_loss_free:    {[f'{u:.3f}' for u in util_free[-1].tolist()]}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 6), sharex=True)
for col, (name, util, loss) in enumerate([
    ("no balancing",        util_none, loss_none),
    ("aux_loss",            util_aux,  loss_aux ),
    ("aux_loss_free",       util_free, loss_free),
]):
    # Top row: utilization per expert over training
    ax = axes[0, col]
    for i in range(util.shape[1]):
        ax.plot(util[:, i], alpha=0.7, label=f"E{i}")
    ax.axhline(1/8, color="black", linestyle="--", alpha=0.5, label="target = 1/N")
    ax.set_ylim(0, 1)
    ax.set_title(f"{name}: expert utilization")
    ax.set_ylabel("fraction of tokens")
    if col == 0:
        ax.legend(loc="upper right", fontsize=7, ncol=2)

    # Bottom row: loss
    ax = axes[1, col]
    ax.plot(loss, color="C3")
    ax.set_yscale("log")
    ax.set_xlabel("step")
    ax.set_ylabel("loss (log)")
    ax.set_title(f"{name}: training loss")

plt.tight_layout()
plt.show()

Three things to read off these plots:

1. **No balancing** (left column): utilization drifts away from the $1/N$ target. Max/min utilization is ~2× across experts — one expert sees twice as many tokens as another. At 400 steps on this toy task it's an annoying imbalance, not a catastrophic collapse. At frontier scale (millions of steps, much noisier data) the compounded effect is dramatic — the seed of the problem is what's visible here.
2. **Aux-loss** (middle column): utilization stays tight around the target via the gradient pressure from $\alpha N \sum_i f_i P_i$. The auxiliary loss trades off against the main loss, which you can sometimes see as a slight bump in the main loss curve.
3. **Aux-loss-free** (right column): utilization is tightest of all — the bias-update rule operates directly on the observed imbalance, with no gradient path involved. The main loss converges cleanly because no auxiliary signal is fighting it.

This is the experimental story that drove DeepSeek-V3's choice. At toy scale the differences are modest; at frontier scale, where small biases compound over hundreds of thousands of optimizer steps, the gap between "uniform-by-construction" and "uniform-by-loss-pressure" matters a lot.

## 3. Sparsity — total vs active parameters

The whole reason MoE is interesting at scale: total params scale with the number of experts, but active per-token compute scales only with $k$. Let's see this concretely for a few configurations.

In [ ]:
def moe_param_counts(d_model, n_experts, top_k, d_ffn_expert, n_shared=1):
    """Approximate FFN-side params for an MoE layer. Ignores router and norms
    (they're a rounding error compared to expert params).
    """
    # Each expert has 3 matrices, each d_model * d_ffn_expert.
    per_expert = 3 * d_model * d_ffn_expert
    total = n_experts * per_expert + (n_shared * per_expert if n_shared else 0)
    active = top_k * per_expert + (n_shared * per_expert if n_shared else 0)
    return total, active

configs = [
    # (label,                    d_model, N,    k, d_ffn_exp, shared)
    ("Vanilla SwiGLU (dense)",   4096,    1,    1, int(4096 * 8 / 3), 0),
    ("Mixtral-8x7B-style",       4096,    8,    2, 14336, 0),
    ("DeepSeek-V2-style",        5120,    160,  6, 1408, 1),
    ("DeepSeek-V3-style",        7168,    256,  8, 2048, 1),
]

print(f"{'config':30s}  {'total FFN (B)':>14s}  {'active FFN (B)':>14s}  {'sparsity':>10s}")
for label, d, N, k, dff, sh in configs:
    total, active = moe_param_counts(d, N, k, dff, sh)
    sparsity = total / active
    print(f"{label:30s}  {total/1e9:>13.2f}B  {active/1e9:>13.2f}B  {sparsity:>9.1f}x")

Read this carefully:

- A dense SwiGLU FFN at $d=4096$ is ~0.13B params, all active.
- A Mixtral-style MoE is 1.4B total but 0.4B active per token (~3.6× sparsity).
- A DeepSeek-V3-style MoE is 11B total but 0.4B active (~25× sparsity).

These are per-layer numbers; the *model* numbers (DeepSeek-V3 is 671B total / 37B active) come from stacking ~60 of these layers. The key message: **DeepSeek-V3's design gets 25× more knowledge capacity for the same active compute as a Mixtral-style design.** The cost is more total parameter memory to hold across the cluster, more all-to-all bandwidth for expert parallelism, and the load-balancing apparatus we built above.

This is the architectural shape every 2026 frontier MoE is converging on.

## 4. MoE transformer block end-to-end

Compose an `MoETransformerBlock` with GQA attention from Module 04. This is the block Module 11 would assemble into a full model if we trained an MoE pretraining run.

In [ ]:
B, T, d_model = 2, 32, 256
n_heads = 8

attn = GroupedQueryAttention(d_model=d_model, n_heads=n_heads, n_kv_heads=2)
moe = MoEFFN(
    d_model=d_model, n_experts=16, top_k=2, d_ffn_expert=128,
    n_shared_experts=1, balancing="aux_loss_free",
)
blk = MoETransformerBlock(d_model=d_model, attention=attn, moe=moe)

freqs_cis = precompute_freqs_cis(d_model // n_heads, T)
x = torch.randn(B, T, d_model)

out, router_info = blk(x, freqs_cis)
print(f"input  shape: {tuple(x.shape)}")
print(f"output shape: {tuple(out.shape)}")

n_attn = sum(p.numel() for p in blk.attn.parameters())
n_moe = sum(p.numel() for p in blk.moe.parameters())
n_total = sum(p.numel() for p in blk.parameters())
print(f"\nparam breakdown:")
print(f"  attention:    {n_attn/1e3:>8.1f}k")
print(f"  MoE FFN:      {n_moe/1e3:>8.1f}k  (16 routed + 1 shared experts)")
print(f"  norms etc:    {(n_total - n_attn - n_moe)/1e3:>8.1f}k")
print(f"  total:        {n_total/1e3:>8.1f}k")

print(f"\nrouter util this batch: {[f'{u:.3f}' for u in router_info.utilization.tolist()]}")
print(f"target:                {2/16:.3f}")

## Recap

You now have:

- `Expert`, `Router`, and `MoEFFN` implementations that mirror DeepSeek-V3's design (fine-grained routed experts + a shared expert).
- A working **aux-loss-free balancing** mechanism — the bias-update rule that operates outside the gradient graph.
- Empirical evidence that aux-loss-free prevents the load-imbalance collapse mode AND converges faster than classical aux-loss balancing on a controlled task.
- A composable `MoETransformerBlock` that drops in as a replacement for Module 05's dense block.
- A concrete sense of the sparsity-ratio tradeoff across frontier MoE configs (Mixtral → DeepSeek-V2 → V3).

**Next:** [Module 07 — Assembling the Full Model](../07-full-model/). Token embeddings, weight tying, muP initialization, and shape tests. Where Module 04's attention, Module 05's block, and Module 06's MoE come together as a single `nn.Module` you can run forward on a real batch.